<a href="https://colab.research.google.com/github/shahzaibali-edu/Road-Accident-Hotspot-Miner/blob/main/Hotspot%20Miner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", palette="deep")

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/Data/US_Accidents.parquet'

# 2. Load only the columns relevant to our mining pipeline to keep peak RAM low
selected_cols = [
    # Identifiers & Target
    'ID', 'Severity',
    # Temporal & Spatial
    'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng',
    'Distance(mi)', 'City', 'County', 'State', 'Timezone', 'Country',
    # Weather Conditions
    'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)',
    'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)',
    'Weather_Condition',
    # Road Features (Binary Flags)
    'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit',
    'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming',
    'Traffic_Signal', 'Turning_Loop',
    # Light Conditions & Free Text (for optional Expert Module)
    'Sunrise_Sunset', 'Civil_Twilight', 'Description'
]

if os.path.exists(file_path):
    print(f"Loading dataset from shared Google Drive Parquet: {file_path}")
    raw_df = pd.read_parquet(file_path, columns=selected_cols)
else:
    print("Shared Drive Parquet not found. Downloading from original Kaggle source...")
    import kagglehub
    kaggle_path = kagglehub.dataset_download("sobhanmoosavi/us-accidents")
    csv_file = os.path.join(kaggle_path, "US_Accidents_March23.csv")
    raw_df = pd.read_csv(csv_file, usecols=selected_cols, low_memory=False)

print(f"Raw Dataset Shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
print("\nRaw Severity Class Distribution (%):")
print((raw_df['Severity'].value_counts(normalize=True).sort_index() * 100).round(2))

# 3. Deliberate Stratified Sampling to 500,000 rows (Lecture 5, Slide 14)
SAMPLE_SIZE = 500_000
if len(raw_df) > SAMPLE_SIZE:
    df, _ = train_test_split(
        raw_df,
        train_size=SAMPLE_SIZE,
        stratify=raw_df['Severity'],
        random_state=RANDOM_SEED
    )
    df = df.reset_index(drop=True)
else:
    df = raw_df.copy()

# Free raw_df from Colab RAM immediately
del raw_df
gc.collect()

print(f"\nStratified Sample Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("Sampled Severity Class Distribution (%):")
print((df['Severity'].value_counts(normalize=True).sort_index() * 100).round(2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading dataset from shared Google Drive Parquet: /content/drive/MyDrive/Data/US_Accidents.parquet
Raw Dataset Shape: 7,728,394 rows × 39 columns

Raw Severity Class Distribution (%):
Severity
1     0.87
2    79.67
3    16.81
4     2.65
Name: proportion, dtype: float64

Stratified Sample Shape: 500,000 rows × 39 columns
Sampled Severity Class Distribution (%):
Severity
1     0.87
2    79.67
3    16.81
4     2.65
Name: proportion, dtype: float64


In [3]:
# 1. Drop post-accident target-leakage columns and zero-variance constant columns
leakage_and_constant_cols = ['End_Time', 'End_Lat', 'End_Lng', 'Country', 'Turning_Loop']
df = df.drop(columns=[c for c in leakage_and_constant_cols if c in df.columns])
print(f"Dropped leakage & constant columns: {leakage_and_constant_cols}")
print(f"Remaining columns: {df.shape[1]}")

# 2. Programmatic Data Dictionary & Attribute Type Summary Table
attribute_type_map = {
    'ID': ('Nominal (Identifier)', 'Discrete', 'None (Row ID only)'),
    'Severity': ('Ordinal (Target 1-4)', 'Discrete', 'Mode, Median, Class %'),
    'Start_Time': ('Interval (Timestamp)', 'Continuous', 'Time extraction, Range'),
    'Start_Lat': ('Interval (Spatial Coord)', 'Continuous', 'Haversine Distance, Median'),
    'Start_Lng': ('Interval (Spatial Coord)', 'Continuous', 'Haversine Distance, Median'),
    'Distance(mi)': ('Ratio (Post-impact length)', 'Continuous', 'Median, IQR, Mean, SD (EDA only)'),
    'City': ('Nominal', 'Discrete', 'Mode, Frequency'),
    'County': ('Nominal', 'Discrete', 'Mode, Frequency'),
    'State': ('Nominal', 'Discrete', 'Mode, Frequency'),
    'Timezone': ('Nominal', 'Discrete', 'Mode, Frequency'),
    'Temperature(F)': ('Interval', 'Continuous', 'Mean, SD, Median, IQR'),
    'Wind_Chill(F)': ('Interval', 'Continuous', 'Mean, SD, Median, IQR'),
    'Humidity(%)': ('Ratio', 'Continuous', 'Mean, SD, Median, IQR'),
    'Pressure(in)': ('Ratio', 'Continuous', 'Mean, SD, Median, IQR'),
    'Visibility(mi)': ('Ratio', 'Continuous', 'Median, IQR, Mean, SD'),
    'Wind_Direction': ('Nominal', 'Discrete', 'Mode, Frequency'),
    'Wind_Speed(mph)': ('Ratio', 'Continuous', 'Median, IQR, Mean, SD'),
    'Precipitation(in)': ('Ratio', 'Continuous', 'Median, IQR, Mean, SD'),
    'Weather_Condition': ('Nominal', 'Discrete', 'Mode, Frequency'),
    'Sunrise_Sunset': ('Binary (Symmetric)', 'Discrete', 'Mode, Proportions'),
    'Civil_Twilight': ('Binary (Symmetric)', 'Discrete', 'Mode, Proportions'),
    'Description': ('Text (Unstructured)', 'Discrete', 'TF-IDF / Token Frequency')
}

# Add all remaining binary road flags automatically
for col in df.columns:
    if col not in attribute_type_map:
        attribute_type_map[col] = ('Binary (Asymmetric Road Flag)', 'Discrete', 'Positive Rate (% True), Mode')

dict_rows = []
for col in df.columns:
    attr_type, disc_cont, valid_stats = attribute_type_map.get(col, ('Nominal', 'Discrete', 'Mode'))
    dict_rows.append({
        'Column': col,
        'Pandas Dtype': str(df[col].dtype),
        'Attribute Type (Lec 2)': attr_type,
        'Discrete/Continuous': disc_cont,
        'Missing (%)': round(df[col].isnull().mean() * 100, 2),
        'Unique Values': df[col].nunique(dropna=True),
        'Valid Summary Operations': valid_stats
    })

data_dictionary_df = pd.DataFrame(dict_rows)
display(data_dictionary_df)

Dropped leakage & constant columns: ['End_Time', 'End_Lat', 'End_Lng', 'Country', 'Turning_Loop']
Remaining columns: 34


,Column,Pandas Dtype,Attribute Type (Lec 2),Discrete/Continuous,Missing (%),Unique Values,Valid Summary Operations
0,ID,object,Nominal (Identifier),Discrete,0.00,500000,None (Row ID only)
1,Severity,int64,Ordinal (Target 1-4),Discrete,0.00,4,"Mode, Median, Class %"
2,Start_Time,object,Interval (Timestamp),Continuous,0.00,486993,"Time extraction, Range"
3,Start_Lat,float64,Interval (Spatial Coord),Continuous,0.00,369627,"Haversine Distance, Median"
4,Start_Lng,float64,Interval (Spatial Coord),Continuous,0.00,370845,"Haversine Distance, Median"
5,Distance(mi),float64,Ratio (Post-impact length),Continuous,0.00,10585,"Median, IQR, Mean, SD (EDA only)"
6,City,object,Nominal,Discrete,0.01,9534,"Mode, Frequency"
7,County,object,Nominal,Discrete,0.00,1593,"Mode, Frequency"
8,State,object,Nominal,Discrete,0.00,49,"Mode, Frequency"
9,Timezone,object,Nominal,Discrete,0.09,4,"Mode, Frequency"


In [4]:
# 1. Parse Start_Time safely (handles mixed fractional seconds in US Accidents 2023)
df['Start_Time'] = pd.to_datetime(df['Start_Time'], format='mixed', errors='coerce')
invalid_time_rows = df['Start_Time'].isnull().sum()
if invalid_time_rows > 0:
    df = df.dropna(subset=['Start_Time']).reset_index(drop=True)
print(f"Dropped {invalid_time_rows} rows with unparseable Start_Time.")

# 2. Uniqueness Check: Exact ID duplicates & repeated event keys
exact_dups = df.duplicated(subset=['ID']).sum()
event_dups = df.duplicated(subset=['Start_Time', 'Start_Lat', 'Start_Lng', 'Severity']).sum()
print(f"Exact ID duplicates found: {exact_dups:,}")
print(f"Repeated event duplicates (same Time, Lat, Lng, Severity) found: {event_dups:,}")

df = df.drop_duplicates(subset=['ID'], keep='first')
df = df.drop_duplicates(subset=['Start_Time', 'Start_Lat', 'Start_Lng', 'Severity'], keep='first').reset_index(drop=True)
print(f"Shape after deduplication: {df.shape[0]:,} rows × {df.shape[1]} columns")

# 3. Consistency Check: Harmonize Wind_Direction & Weather_Condition strings
wind_map = {
    'Calm': 'CALM', 'North': 'N', 'South': 'S', 'East': 'E', 'West': 'W',
    'Variable': 'VAR', 'NNW': 'NW', 'WNW': 'NW', 'NNE': 'NE', 'ENE': 'NE',
    'SSW': 'SW', 'WSW': 'SW', 'SSE': 'SE', 'ESE': 'SE'
}
df['Wind_Direction'] = df['Wind_Direction'].astype(str).str.strip().replace({'nan': np.nan, 'None': np.nan})
df['Wind_Direction'] = df['Wind_Direction'].replace(wind_map)

def standardize_weather(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if any(k in s for k in ['thunder', 't-storm', 'storm', 'hail', 'tornado', 'squall']):
        return 'Thunderstorm'
    if any(k in s for k in ['snow', 'sleet', 'ice', 'freezing', 'wintry', 'blizzard']):
        return 'Snow_Ice'
    if any(k in s for k in ['rain', 'drizzle', 'shower', 'precipitation']):
        return 'Rain'
    if any(k in s for k in ['fog', 'mist', 'haze', 'smoke']):
        return 'Fog_Low_Vis'
    if any(k in s for k in ['wind', 'dust', 'sand', 'blowing']):
        return 'Windy_Dust'
    if any(k in s for k in ['cloud', 'overcast', 'scattered', 'mostly']):
        return 'Cloudy'
    if any(k in s for k in ['clear', 'fair']):
        return 'Clear'
    return 'Other'

df['Weather_Category'] = df['Weather_Condition'].apply(standardize_weather)
print("\nHarmonized Weather_Category counts (before missing-value imputation):")
print(df['Weather_Category'].value_counts(dropna=False))

# 4. Validity Check: Flag physically impossible sensor readings and set to NaN
validity_rules = {
    'Temperature(F)': (-60.0, 140.0),
    'Humidity(%)': (0.0, 100.0),
    'Pressure(in)': (15.0, 45.0),
    'Visibility(mi)': (0.0, 100.0),
    'Wind_Speed(mph)': (0.0, 200.0),
    'Precipitation(in)': (0.0, 50.0),
    'Distance(mi)': (0.0, 500.0)
}

validity_log = []
for col, (low, high) in validity_rules.items():
    invalid_mask = (df[col] < low) | (df[col] > high)
    n_invalid = int(invalid_mask.sum())
    if n_invalid > 0:
        df.loc[invalid_mask, col] = np.nan
    validity_log.append({'Column': col, 'Valid Range': f"[{low}, {high}]", 'Invalid Values Replaced with NaN': n_invalid})

display(pd.DataFrame(validity_log))

Dropped 0 rows with unparseable Start_Time.
Exact ID duplicates found: 0
Repeated event duplicates (same Time, Lat, Lng, Severity) found: 2,886
Shape after deduplication: 497,114 rows × 34 columns

Harmonized Weather_Category counts (before missing-value imputation):
Weather_Category
Clear           216861
Cloudy          200348
Rain             33093
Fog_Low_Vis      12986
Snow_Ice         11573
NaN              11251
Thunderstorm      5773
Windy_Dust        5229
Name: count, dtype: int64


,Column,Valid Range,Invalid Values Replaced with NaN
0,Temperature(F),"[-60.0, 140.0]",3
1,Humidity(%),"[0.0, 100.0]",0
2,Pressure(in),"[15.0, 45.0]",3
3,Visibility(mi),"[0.0, 100.0]",0
4,Wind_Speed(mph),"[0.0, 200.0]",0
5,Precipitation(in),"[0.0, 50.0]",0
6,Distance(mi),"[0.0, 500.0]",0
